# Task 5a + 5b: generated-answer leakage and known-pairs attacker
Runtime → Change runtime type → **T4 GPU**. Upload `answer_generation.py`, `known_pairs_attack.py` and the CURRENT `secret_ushape.py` (the one with `save_adapters`) to `/content` (folder icon on the left).

**Prerequisite:** trained adapters saved by `secret_ushape.py` (run once in `run_multisplit_colab.ipynb`):
```
!python secret_multisplit.py --points "4,2" --seeds 0 --save_dir {SAVE}
!python secret_multisplit.py --points "2,2" --seeds 1 --save_dir {SAVE}
```
They write `adapters_lp0.1_la{0,0.01,0.03,0.1}.pt` into `{SAVE}/runs/H4_T2_seed0/` and `{SAVE}/runs/H2_T2_seed1/`.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
!pip install -q datasets
from google.colab import drive
drive.mount('/content/drive')
SAVE = '/content/drive/MyDrive/multisplit'
OUT = SAVE + '/task5'
import os, glob
os.makedirs(OUT, exist_ok=True)
RUNS = {'H4_T2': SAVE + '/runs/H4_T2_seed0', 'H2_T2': SAVE + '/runs/H2_T2_seed1'}
for k, d in RUNS.items():
    print(k, sorted(os.path.basename(p) for p in glob.glob(d + '/adapters_*.pt')))

## Code test (CPU, tiny random model, a few minutes) — optional

In [ ]:
!python answer_generation.py --smoke --out /content/smoke_5a
!python known_pairs_attack.py --smoke --out /content/smoke_5b

## 5a: generated-answer leakage, all four λa per split (run time on a T4 not measured yet)

In [ ]:
for k, d in RUNS.items():
    files = ' '.join(sorted(glob.glob(d + '/adapters_*.pt')))
    !python answer_generation.py --adapters {files} --out {OUT}/5a_{k}

## 5b: known-pairs attacker (k = 10, 100, 1000), λa = 0.03 per split (per split: train A, 6 fine-tunes, train B; run time not measured yet)
Run one split per session if time is short; results are written to Drive.

In [ ]:
LA = '0.03'
for k, d in RUNS.items():
    f = f'{d}/adapters_lp0.1_la{LA}.pt'
    !python known_pairs_attack.py --adapters {f} --out {OUT}/5b_{k}_la{LA}

## Download

In [ ]:
!cd {OUT} && zip -qr /content/task5_results.zip .
from google.colab import files
files.download('/content/task5_results.zip')